# AEI last version
$(\omega - m\Omega_\theta)^2 = \Omega_r^2 + \frac{2B_0 |k|}{\Sigma r} + \frac{k^2 c_s^2}{r^2}$

- $B_0$ campo imperturbato, prpendicolare al disco. legato al materiale del disco - assumiamo legge di potenza con esponente 5/4 come da Shakura&Sunyaev
- $\Sigma$ densità superficiale, legge di potenza con esponente ..
- assumo disco sottile, con aspect ratio $H/r = HOR$
- $c_s$ velocità del suono - in disco sottile è dato approssimativamente dalla formula sotto
- k = numero d'onda. lo prendiamo come parametro liberto ma poi controlliamo che stia in range ragionevole ($k \approx 1/H$ con H scala verticale tipica del disco)
- m: generalmente il modo eccitato ha un solo braccio, perciò poniamo m=1 per ridurre il numro di parametri


> $B_0 = B_{00} (r/r_{H})^{-\alpha_B}$. $B_{00}$ valore di riferimento sull'orizzonte degli eventi ($r_H$) del BH

> $\Sigma = \Sigma_0 (r/r_{in})^{-\alpha_\Sigma}$. $\Sigma_0$ riferimento al borndo interno del disco

> $c_s \approx (H/r) v_\varphi = (H/r) r \Omega_\varphi$

> $0.1/r < |k| < 10/r \ ,\ $ 
> $\beta = \frac{ 8 \pi p}{B^2_0} \approx \frac{ 8 \pi \Sigma c_s^2}{B^2_0} \leq 1 \ ,\ $
> $\frac{d}{dr}\left( \frac{\Omega\Sigma}{B^2_0} \right) > 0$

## settings

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from aei_2.aei_setup import (
    solve_k_aei, compute_beta, compute_dQdr,
    r_ilr, r_olr, r_corotation, _make_interp,
    make_disk, get_resonances
)

from setup import M_BH, NU0, r_isco, Rg_SUN, set_style, fix_spines
from aei_2.simple_disc import disk_model_simple

In [ ]:
from matplotlib.colors import LogNorm
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec

set_style()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════
A_GRID      = np.linspace(-1, 1, 100)   # spin grid
N_R        = 300   # radial points for full profile (ISCO → OLR)
N_R_CAV    = 200   # radial points for cavity (ISCO → ILR)

# Reference values for dQ/dr plots (only sign matters, not amplitude)
B00_REF    = 1e4
SIGMA0_REF = 1e4

## simple

In [ ]:
# SIMPLE ====================================================================

ALP_B = 5 / 4    # Simple-v1 magnetic field exponent
ALP_S = 3 / 5    # Simple-v1 surface density exponent

CONFIGS = [
    dict(mm=1, hor=0.05,  color='#3b82f6', ls='-',  label=r'$m=1,\ H/r=0.05$'),
    dict(mm=1, hor=0.001, color='#f97316', ls='-',  label=r'$m=1,\ H/r=10^{-3}$'),
    dict(mm=2, hor=0.05,  color='#22c55e', ls='--', label=r'$m=2,\ H/r=0.05$'),
    dict(mm=2, hor=0.001, color='#ef4444', ls='--', label=r'$m=2,\ H/r=10^{-3}$'),
]

B00_GRID    = np.logspace(1,  8, 48)          # B00 [G]
SIGMA0_GRID = np.logspace(2,  7, 36)          # Sigma0 [g/cm²]

In [ ]:
def dQdr_profile(r_vec, a, B00, Sigma0, hor):
    """
    Compute dQ/dr = d/dr [Omega_phi * Sigma / B0²].
    The sign is independent of B00, Sigma0 (positive multiplicative constant).
    H/r (hor) does not enter Q at all.
    """
    res = make_disk(disk_model_simple, (r_vec, a, B00, Sigma0, ALP_B, ALP_S, hor))
    B0i = _make_interp(r_vec, res[0])
    Si  = _make_interp(r_vec, res[1])
    return compute_dQdr(r_vec, a, B0i, Si, M_BH)

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── MAIN SCAN
# ═══════════════════════════════════════════════════════════════════════════════

def cavity_scan(mm, hor, verbose=False):
    """
    Three-step filter over the full (a, B00, Sigma0) grid.

    Step A  dQ/dr > 0 for ALL r ∈ [CR-eps, CR+eps]
            Computed ONCE per (a, mm) — independent of B00, Sigma0, hor.
            The B00/Sigma0 loop is entirely skipped for failing spins.

    Step B  beta(r) ≤ 1 for ALL r ∈ [ISCO, OLR]
            Depends on Sigma0/B00² and hor.

    Step C  1 ≤ k(r) ≤ 1/hr(r) for ALL r ∈ [ISCO, ILR]
            Solves the AEI dispersion relation point by point.

    Returns
    -------
    pd.DataFrame with one row per (a, B00, Sigma0).
    Columns: a, B00, Sigma0, r_ILR, r_OLR, r_CR,
             dQdr_CR, pass_shear, pass_beta, pass_wkb,
             beta_max, k_med, k_min_val, k_max_val, frac_wkb_ok
    """
    records = []
    n_spin = len(A_GRID)
    n_par  = len(B00_GRID) * len(SIGMA0_GRID)

    for i_a, a_val in enumerate(A_GRID):
        rISCO = float(r_isco(a_val))
        r_ILR, r_OLR, r_CR = get_resonances(a_val, mm)

        if verbose:
            print(f'  spin {i_a+1}/{n_spin}  a={a_val:.2f} '
                  f'| r_ILR={r_ILR:.1f}  r_OLR={r_OLR:.1f}  r_CR={r_CR:.1f}',
                  end='  ')

        # ── sanity check on resonance radii ───────────────────────────────
        if not (np.isfinite(r_OLR) and np.isfinite(r_ILR)):
            if verbose:
                print('SKIP (resonances not found)')
            # fill records with failed rows
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=np.nan, r_OLR=np.nan, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        if r_ILR <= rISCO * 1.001:
            if verbose:
                print('SKIP (ILR inside ISCO)')
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── Step A: dQ/dr (independent of B00, Sigma0, hor) ───────────────
        r_olr_cap = min(r_OLR, rISCO * 500.0)
        r_full    = np.geomspace(rISCO*1.001, r_olr_cap, N_R)

        r_cr_eps = np.linspace(r_CR*0.95, r_CR*1.05, 20)
        dq_full  = dQdr_profile(r_cr_eps, a_val, B00_REF, SIGMA0_REF, hor)
        dqdr_at_CR = float(np.interp(r_CR, r_cr_eps, dq_full))
        pass_shear = bool(dqdr_at_CR > 0)

        if verbose:
            shear_str = 'SHEAR OK' if pass_shear else 'shear FAIL'
            print(shear_str)

        if not pass_shear:
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── radial grids for beta & cavity ─────────────────────────────────
        r_ilr_cap = min(r_ILR*0.98, r_olr_cap)
        r_cavity  = np.geomspace(rISCO*1.01, r_ilr_cap, N_R_CAV)

        # ── Steps B & C: loop over (B00, Sigma0) ──────────────────────────
        for B00 in B00_GRID:
            for S0 in SIGMA0_GRID:

                # ── Step B: beta ≤ 1 in [ISCO, OLR] ─────────────────────
                res_full = make_disk(disk_model_simple, (r_full, a_val, B00, S0, ALP_B, ALP_S, hor))
                hr_full  = res_full[3]   # per-point H/r from model
                beta_full = compute_beta(res_full[0], res_full[1], res_full[2],
                                         r_full, hr_full, M_BH)
                pass_beta = bool(np.all(beta_full <= 1.0))
                beta_max  = float(np.max(beta_full))

                if not pass_beta:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=True, pass_beta=False,
                                        pass_wkb=False, beta_max=beta_max))
                    continue

                # ── Step C: WKB in cavity [ISCO, ILR] ────────────────────
                res_cav  = make_disk(disk_model_simple, (r_cavity, a_val, B00, S0, ALP_B, ALP_S, hor))
                B0_c, Sg_c, cs_c, hr_c = res_cav[:4]

                k_arr = solve_k_aei(r_cavity, a_val, B0_c, Sg_c, cs_c,
                                    m=mm, M=M_BH)
                k_max_arr = 1 / np.maximum(hr_c, 1e-10)  # 1/hr(r)

                wkb_ok   = (np.isfinite(k_arr)
                            & (k_arr >= mm)
                            & (k_arr <= k_max_arr))
                pass_wkb = bool(wkb_ok.mean() >= 0.95)
                frac_ok  = float(wkb_ok.sum() / len(wkb_ok))

                k_valid = k_arr[np.isfinite(k_arr)]
                records.append(dict(
                    a=a_val, B00=B00, Sigma0=S0,
                    r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                    dQdr_CR=dqdr_at_CR,
                    pass_shear=True, pass_beta=True, pass_wkb=pass_wkb,
                    beta_max=beta_max,
                    frac_wkb_ok=frac_ok,
                    k_med=float(np.nanmedian(k_arr)),
                    k_min_val=float(np.nanmin(k_arr)) if k_valid.size else np.nan,
                    k_max_val=float(np.nanmax(k_arr)) if k_valid.size else np.nan,
                ))

    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── RUN ALL CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

def run_all():
    """Run cavity_scan for all 4 (m, H/r) configurations."""
    all_results = {}
    for cfg in CONFIGS:
        mm, hor = cfg['mm'], cfg['hor']
        print(f"\n{'='*65}")
        print(f"  Config: m={mm}, H/r={hor}")
        print(f"{'='*65}")
        df = cavity_scan(mm, hor, verbose=False)
        all_results[(mm, hor)] = df

        n      = len(df)
        n_sh   = df['pass_shear'].sum()
        n_be   = df.get('pass_beta',  pd.Series(dtype=bool)).sum() if 'pass_beta' in df else 0
        n_wkb  = df.get('pass_wkb',   pd.Series(dtype=bool)).sum() if 'pass_wkb'  in df else 0
        print(f'\n  Total combos : {n}')
        print(f'  Pass shear   : {n_sh:>6}  ({100*n_sh/n:.1f}%)')
        print(f'  + beta ≤ 1   : {n_be:>6}  ({100*n_be/n:.1f}%)')
        print(f'  + WKB cavity : {n_wkb:>6}  ({100*n_wkb/n:.1f}%)')

    return all_results

#### calc

In [ ]:
# ── step 1: full scan ─────────────────────────────────────────────────────
print('\n> Running cavity scan for all configs...')
all_results = run_all()

#### plot 1

In [ ]:
cfg = CONFIGS[1]
mm, hor = cfg['mm'], cfg['hor']
df = all_results[(mm, hor)]

stages = [
    ('pass_shear', r'$dQ/dr > 0$ at $r_{\rm CR}$'),
    ('pass_beta',  r'$+\ \beta \leq 1$ everywhere'),
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(7, 2.2))
gs = gridspec.GridSpec(1, 4, figure=fig,
                    width_ratios=[1, 1, 1, 0.08],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
cax  = fig.add_subplot(gs[0, 3])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    im = ax.pcolormesh(SIGMA0_GRID, B00_GRID, frac_map,
                        vmin=0, vmax=1, cmap='RdYlGn', shading='auto')

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.85)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.85)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=3)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')
    ax.set_title(title)

    # Nascondi le etichette y dei pannelli interni
    if col != 'pass_shear':
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/simple_heatmap_{hor}_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

#### plot 2

In [ ]:
mm=2
cfgs_m = [c for c in CONFIGS if c['mm'] == mm]

fig = plt.figure(figsize=(7,2.8))
gs = gridspec.GridSpec(1, 3, figure=fig,
                    width_ratios=[1, 1, 0.05],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
cax  = fig.add_subplot(gs[0, 2])   # asse dedicato alla colorbar

# shared colour scale 0→1
vmin, vmax = 0.0, 1.0

for ax, cfg in zip(axes, cfgs_m):
    fix_spines(ax)
    hor = cfg['hor']
    key = (mm, hor)

    if key not in all_results:
        ax.text(0.5, 0.5, f'Dati mancanti\n(mm={mm}, hor={hor})',
                ha='center', va='center', transform=ax.transAxes)
        continue

    df = all_results[key]

    # ── build heatmap ──────────────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            frac_map[i, j] = sub['pass_wkb'].sum() / len(sub)

    n_valid_cells = int(np.sum(frac_map > 0))
    n_cells       = frac_map.size

    im = ax.pcolormesh(
        SIGMA0_GRID, B00_GRID, frac_map,
        vmin=vmin, vmax=vmax,
        cmap='RdYlGn', shading='auto',
    )

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.9)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.9)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')

    ax.text(0.97, 0.03,
        rf'$H/r = {hor}$',
        transform=ax.transAxes, va='bottom',
        horizontalalignment='right',
        color='black',
        bbox=dict(boxstyle='round', fc='white', alpha=0.5))

    # Nascondi le etichette y dei pannelli interni
    if hor == 0.001:
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/wkb_heatmap_comparison.{ext}', bbox_inches='tight', dpi=150)
plt.show()

## NT

In [ ]:
from aei_2.nt_disc import disk_model_NT, disk_inner_values_NT

In [ ]:
CONF_nT = [
    dict(mm=1, color='#3b82f6', ls='--',  label=r'$m=1$'),
    dict(mm=2,color='#ef4444', ls='.', label=r'$m=2$'),
]

MDOT_SCALE_NT = 10    # eta=0.1
# Parameter grids
N_MDOT  = 24
N_ALPHA = 18
MDOT_PLOT  = np.logspace(-5, 0, N_MDOT)   # physical mdot
MDOT_GRID = MDOT_PLOT * MDOT_SCALE_NT                 # mdot grid for NT disc (scaled up)
ALPHA_GRID = np.logspace(-3, 0, N_ALPHA)                    # alpha_visc

In [ ]:
def dQdr_nt(r_vec, a, mdot, alpha):
    """
    Compute dQ/dr = d/dr [Omega_phi * Sigma / B0²].
    The sign is independent of B00, Sigma0 (positive multiplicative constant).
    H/r (hor) does not enter Q at all.
    """
    res = make_disk(disk_model_NT, (r_vec, a, mdot, alpha))
    B0i = _make_interp(r_vec, res[0])
    Si  = _make_interp(r_vec, res[1])
    return compute_dQdr(r_vec, a, B0i, Si, M_BH)

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── MAIN SCAN
# ═══════════════════════════════════════════════════════════════════════════════

def cavity_nt(mm, verbose=False):
    records = []
    n_spin = len(A_GRID)
    n_par  = len(MDOT_GRID) * len(ALPHA_GRID)

    for i_a, a_val in enumerate(A_GRID):
        rISCO = float(r_isco(a_val))
        r_ILR, r_OLR, r_CR = get_resonances(a_val, mm)

        if verbose:
            print(f'  spin {i_a+1}/{n_spin}  a={a_val:.2f} '
                  f'| r_ILR={r_ILR:.1f}  r_OLR={r_OLR:.1f}  r_CR={r_CR:.1f}',
                  end='  ')

        # ── sanity check on resonance radii ───────────────────────────────
        if not (np.isfinite(r_OLR) and np.isfinite(r_ILR)):
            if verbose:
                print('SKIP (resonances not found)')
            # fill records with failed rows
            for mdot in MDOT_GRID:
                for alpha in ALPHA_GRID:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=np.nan, r_OLR=np.nan, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        if r_ILR <= rISCO * 1.001:
            if verbose:
                print('SKIP (ILR inside ISCO)')
            for mdot in MDOT_GRID:
                for alpha in ALPHA_GRID:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ─────────────────
        #r_olr_cap = min(r_OLR, rISCO * 500.0)
        #r_full    = np.geomspace(rISCO*1.001, r_olr_cap, N_R)
        r_ilr_cap = r_ILR*0.98
        r_cavity  = np.geomspace(rISCO*1.01, r_ilr_cap, N_R_CAV)

        # ── loop over (mdot, alpha) ──────────────────────────
        for mdot in MDOT_GRID:
            for alpha in ALPHA_GRID:
                """ no need for this cause always satisfied
                # ── Step A: beta ≤ 1 in [ISCO, OLR] ─────────────────────
                res_full = make_disk(disk_model_NT, (r_full, a_val, mdot, alpha))
                hr_full  = res_full[3]   # per-point H/r from model
                beta_full = compute_beta(res_full[0], res_full[1], res_full[2],
                                         r_full, hr_full, M_BH)
                pass_beta = bool(np.all(beta_full <= 1.0))
                beta_max  = float(np.max(beta_full))

                if not pass_beta:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False, beta_max=beta_max))
                    continue

                # ── Step B: shear at CR ────────────────────
                r_cr_eps = np.linspace(r_CR*0.95, r_CR*1.05, 20)
                dq_full  = dQdr_nt(r_cr_eps, a_val, mdot, alpha)
                dqdr_at_CR = float(np.interp(r_CR, r_cr_eps, dq_full))
                pass_shear = bool(dqdr_at_CR > 0)

                if verbose:
                    shear_str = 'SHEAR OK' if pass_shear else 'shear FAIL'
                    print(shear_str)

                if not pass_shear:
                    for mdot in MDOT_GRID:
                        for alpha in ALPHA_GRID:
                            records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                                r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                                dQdr_CR=dqdr_at_CR,
                                                pass_shear=False, pass_beta=True,
                                                pass_wkb=False))
                    continue

                """
                # ── Step C: WKB in cavity [ISCO, ILR] ────────────────────
                res_cav  = make_disk(disk_model_NT, (r_cavity, a_val, mdot, alpha))
                B0_c, Sg_c, cs_c, hr_c = res_cav[:4]

                k_arr = solve_k_aei(r_cavity, a_val, B0_c, Sg_c, cs_c,
                                    m=mm, M=M_BH)
                k_max_arr = 1 / np.maximum(hr_c, 1e-10)  # 1/hr(r)

                wkb_ok   = (np.isfinite(k_arr)
                            & (k_arr >= mm)
                            & (k_arr <= k_max_arr))
                pass_wkb = bool(np.all(wkb_ok))
                frac_ok  = float(wkb_ok.sum() / len(wkb_ok))

                k_valid = k_arr[np.isfinite(k_arr)]
                records.append(dict(
                    a=a_val, mdot=mdot, alpha=alpha,
                    r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                    dQdr_CR=1, #dqdr_at_CR,
                    pass_shear=True, pass_beta=True, pass_wkb=pass_wkb,
                    beta_max=1, #beta_max,
                    frac_wkb_ok=frac_ok,
                    k_med=float(np.nanmedian(k_arr)),
                    k_min_val=float(np.nanmin(k_arr)) if k_valid.size else np.nan,
                    k_max_val=float(np.nanmax(k_arr)) if k_valid.size else np.nan,
                ))

    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── RUN ALL CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

def run_nt():
    all_results = {}
    for cfg in CONF_nT:
        mm = cfg['mm']
        print(f"\n{'='*65}")
        print(f"  Config: m={mm}")
        print(f"{'='*65}")
        df = cavity_nt(mm, verbose=False)
        all_results[mm] = df

        n      = len(df)
        n_sh   = df['pass_shear'].sum()
        n_be   = df.get('pass_beta',  pd.Series(dtype=bool)).sum() if 'pass_beta' in df else 0
        n_wkb  = df.get('pass_wkb',   pd.Series(dtype=bool)).sum() if 'pass_wkb'  in df else 0
        print(f'\n  Total combos : {n}')
        print(f'  Pass shear   : {n_sh:>6}  ({100*n_sh/n:.1f}%)')
        print(f'  + beta ≤ 1   : {n_be:>6}  ({100*n_be/n:.1f}%)')
        print(f'  + WKB cavity : {n_wkb:>6}  ({100*n_wkb/n:.1f}%)')

    return all_results

#### calc

In [ ]:
# ── step 1: full scan ─────────────────────────────────────────────────────
print('\n> Running cavity scan for all configs...')
all_nt = run_nt()

#### plot 1

In [ ]:
cfg = CONF_nT[0]
mm = cfg['mm']
df = all_nt[mm]

stages = [
    ('pass_shear', r'$dQ/dr > 0$ at $r_{\rm CR}$'),
    ('pass_beta',  r'$+\ \beta \leq 1$ everywhere'),
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(7, 2.2))
gs = gridspec.GridSpec(1, 4, figure=fig,
                    width_ratios=[1, 1, 1, 0.08],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
cax  = fig.add_subplot(gs[0, 3])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(MDOT_GRID), len(ALPHA_GRID)), np.nan)
    for i, mdot in enumerate(MDOT_GRID):
        for j, alpha in enumerate(ALPHA_GRID):
            sub = df[(df['mdot'] == mdot) & (df['alpha'] == alpha)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    X, Y = np.meshgrid(MDOT_PLOT, ALPHA_GRID)
    im = ax.pcolormesh(X, Y, frac_map.T,
                    vmin=0, vmax=1, cmap='RdYlGn', shading='auto')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\dot{m}$')
    ax.set_title(title)

    # Nascondi le etichette y dei pannelli interni
    if col != 'pass_shear':
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$\alpha$')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/nt_mdot_alpha_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
cfg = CONF_nT[0]
mm = cfg['mm']
df = all_nt[mm]

stages = [
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(3.4, 3.1))
gs = gridspec.GridSpec(1, 2, figure=fig,
                    width_ratios=[1, 0.05],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(1)]
cax  = fig.add_subplot(gs[0, 1])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(MDOT_GRID), len(ALPHA_GRID)), np.nan)
    for i, mdot in enumerate(MDOT_GRID):
        for j, alpha in enumerate(ALPHA_GRID):
            sub = df[(df['mdot'] == mdot) & (df['alpha'] == alpha)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    X, Y = np.meshgrid(MDOT_PLOT, ALPHA_GRID)
    im = ax.pcolormesh(X, Y, frac_map.T,
                    vmin=0, vmax=1, cmap='RdYlGn', shading='auto')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\dot{m}$')

    # Nascondi le etichette y dei pannelli interni
    if col != 'pass_shear':
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$\alpha$')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/wkb_nt_mdot_alpha_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

#### plot 2

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.  COLLECT  —  B_ISCO, Sigma_ISCO per row usando lo spin reale
# ═══════════════════════════════════════════════════════════════════════════════

def collect_bisco_sigma_spin(df_nt, stage='pass_wkb', verbose=True):
    """
    For each row in df_nt that passes `stage`, compute B_ISCO and Sigma_ISCO
    using the actual spin a of that row (NOT a fixed pivot).

    Parameters
    ----------
    df_nt  : DataFrame  — one row per (a, mdot, alpha), output of cavity_nt()
    stage  : str        — column name of the boolean filter to apply
                          ('pass_shear', 'pass_beta', 'pass_wkb')
    verbose: bool

    Returns
    -------
    out : pd.DataFrame with columns
        a, mdot, alpha, B_ISCO [G], Sigma_ISCO [g/cm²]
        (only rows where disk_inner_values_NT succeeded)
    """
    # ── filter ────────────────────────────────────────────────────────────
    df_ok = df_nt[df_nt[stage]].copy().reset_index(drop=True)
    total = len(df_ok)

    if verbose:
        print(f"[collect_bisco_sigma_spin]  stage='{stage}'  "
              f"{total} rows to process...")

    records = []
    # cache: same (mdot, alpha, a) triple can repeat if the scan has fine grids
    _cache = {}

    for i, row in df_ok.iterrows():
        a_val   = float(row['a'])
        mdot    = float(row['mdot'])
        alpha   = float(row['alpha'])

        key = (round(a_val, 8), round(mdot, 12), round(alpha, 8))
        if key in _cache:
            B_isco, S_isco = _cache[key]
        else:
            try:
                iv = disk_inner_values_NT(a_val, mdot,
                                          alpha_visc=alpha, M=M_BH)
                B_isco = float(iv['B_ISCO'])
                S_isco = float(iv['Sigma_ISCO'])
            except Exception:
                B_isco, S_isco = np.nan, np.nan
            _cache[key] = (B_isco, S_isco)

        if np.isfinite(B_isco) and np.isfinite(S_isco) and B_isco > 0 and S_isco > 0:
            records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                B_ISCO=B_isco, Sigma_ISCO=S_isco))

        if verbose and (i + 1) % max(1, total // 10) == 0:
            print(f"  {i+1}/{total}  ({100*(i+1)/total:.0f}%)", end='\r')

    print()
    out = pd.DataFrame(records)
    if verbose and not out.empty:
        print(f"  → {len(out)} valid points")
        print(f"  B_ISCO   : [{out['B_ISCO'].min():.2e}, {out['B_ISCO'].max():.2e}] G")
        print(f"  Sig_ISCO : [{out['Sigma_ISCO'].min():.2e}, {out['Sigma_ISCO'].max():.2e}] g/cm²")
        print(f"  spin a   : [{out['a'].min():.2f}, {out['a'].max():.2f}]")
    return out


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  SCATTER PLOT  — colour = spin a
# ═══════════════════════════════════════════════════════════════════════════════

def scatter_bisco_sigma(df_nt, stage='pass_wkb',
                        ref_B=None, ref_S=None,
                        cmap='RdBu_r',
                        alpha_pt=0.55,
                        s_pt=14,
                        figsize=(3.4, 2.9),
                        outdir='aei_2',
                        verbose=True):
    """
    Scatter plot in the (Sigma_ISCO, B_ISCO) plane, colour = spin a.

    Points are NOT binned — each passing (a, mdot, alpha) combination
    is plotted individually, so the spin dependence of B_ISCO and
    Sigma_ISCO is fully visible.

    Parameters
    ----------
    df_nt   : DataFrame   — output of cavity_nt / run_nt()[mm]
    stage   : str         — filter column ('pass_shear'/'pass_beta'/'pass_wkb')
    ref_B   : float|None  — horizontal reference line [G]
    ref_S   : float|None  — vertical reference line [g/cm²]
    cmap    : str         — colourmap for spin (default diverging RdBu_r)
    alpha_pt: float       — point transparency
    s_pt    : float       — point size
    figsize : tuple
    outdir  : str         — directory for saved figures
    verbose : bool

    Returns
    -------
    fig : matplotlib.figure.Figure
    df_plot : pd.DataFrame  — the data that was plotted
    """
    # ── collect ───────────────────────────────────────────────────────────
    df_plot = collect_bisco_sigma_spin(df_nt, stage=stage, verbose=verbose)
    if df_plot.empty:
        print("No valid points to plot.")
        return None, df_plot

    a_vals  = df_plot['a'].values
    S_vals  = df_plot['Sigma_ISCO'].values
    B_vals  = df_plot['B_ISCO'].values

    # colour mapped on spin: symmetric range [-1, 1]
    norm = Normalize(vmin=-1.0, vmax=1.0)
    cm   = plt.get_cmap(cmap)

    # ── figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(S_vals, B_vals,
                    c=a_vals, cmap=cm, norm=norm,
                    s=s_pt, alpha=alpha_pt,
                    linewidths=0, rasterized=True, zorder=3)

    cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cm), ax=ax, pad=0.02)
    cb.set_label(r'Spin  $a$')

    # ── reference lines / cross ───────────────────────────────────────────
    if ref_S is not None:
        ax.axvline(ref_S, color='black', ls='--', lw=1, alpha=0.8)
    if ref_B is not None:
        ax.axhline(ref_B, color='black', ls='--', lw=1, alpha=0.8)
    if ref_S is not None and ref_B is not None:
        ax.plot(ref_S, ref_B, 'kx', ms=8, mew=1.5, zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_{\rm ISCO}$  [g cm$^{-2}$]')
    ax.set_ylabel(r'$B_{\rm ISCO}$  [G]')

    stage_labels = {
        'pass_shear': r'$dQ/dr > 0$',
        'pass_beta':  r'$dQ/dr > 0$ + $\beta \leq 1$',
        'pass_wkb':   r'all checks (+ WKB cavity)',
    }

    plt.tight_layout()

    # ── save ──────────────────────────────────────────────────────────────
    tag = stage.replace('pass_', '')
    for ext in ('pdf', 'png'):
        fpath = f'{outdir}/scatter_bisco_sigma_NT_{tag}.{ext}'
        plt.savefig(fpath, bbox_inches='tight', dpi=150)
        if verbose:
            print(f'  → {fpath}')

    plt.show()
    return fig, df_plot

In [ ]:
fig, df_plot = scatter_bisco_sigma(
    all_nt[1], stage='pass_wkb',
    ref_B=1e4, ref_S=1e4, outdir='aei_2')

#### test

In [ ]:
from aei_2.nt_disc import disk_model_NT, nt_boundaries, H_NT, _Sigma_disk
from setup import nu_phi
def disk_inner_values_NT(a, mdot, alpha_visc, M=M_BH):
    """
    Restituisce B e Sigma valutati esattamente a r_ISCO.
    Nessun raccordo, nessuna griglia fine — un solo punto.
    """
    rISCO = float(r_isco(a))
    r_AB, r_BC, zone_present = nt_boundaries(a, mdot, alpha=alpha_visc, M=M)

    r_pt    = np.array([rISCO])
    S_isco  = float(_Sigma_disk(r_pt, a, mdot, alpha_visc, M, r_AB, r_BC)[0])
    H_isco  = float(H_NT(r_pt, a, mdot, alpha_visc, M, r_AB, r_BC)[0])
    Om_isco = float(2.0 * np.pi * nu_phi(r_pt, a, M)[0])
    B_isco  = float(np.sqrt(max(4.0 * np.pi * S_isco * Om_isco**2 * H_isco, 0.0)))

    return {
        'r_ISCO':       rISCO,
        'r_AB':         r_AB,
        'r_BC':         r_BC,
        'zone_present': zone_present,
        'Sigma_ISCO':   S_isco,
        'B_ISCO':       B_isco,
    }

In [ ]:
def add_isco_values(df_nt):
    """
    Aggiunge B_ISCO e Sigma_ISCO a ogni riga di all_nt[mm].
    Calcola una volta sola per ogni combinazione unica (a, mdot, alpha).
    """
    combos = df_nt[['a', 'mdot', 'alpha']].drop_duplicates()

    rows = []
    for _, row in combos.iterrows():
        iv = disk_inner_values_NT(row['a'], row['mdot'], alpha_visc=row['alpha'])
        rows.append({
            'a': row['a'], 'mdot': row['mdot'], 'alpha': row['alpha'],
            'B_ISCO':     iv['B_ISCO'],
            'Sigma_ISCO': iv['Sigma_ISCO'],
        })

    return df_nt.merge(pd.DataFrame(rows), on=['a', 'mdot', 'alpha'], how='left')


def scatter_bisco_sigma(df, stage='pass_wkb',
                        ref_B=None, ref_S=None,
                        cmap='RdBu_r', figsize=(6, 4.8)):
    """
    Scatter (Sigma_ISCO, B_ISCO) colorato per spin.
    df deve già avere le colonne B_ISCO e Sigma_ISCO (da add_isco_values).
    """
    df_ok = df[df[stage] & df['B_ISCO'].notna()].copy()

    norm = plt.Normalize(-1, 1)
    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(df_ok['Sigma_ISCO'], df_ok['B_ISCO'],
                    c=df_ok['a'], cmap=cmap, norm=norm,
                    s=14, alpha=0.55, linewidths=0, rasterized=True)

    plt.colorbar(sc, ax=ax, label=r'Spin $a$')

    if ref_S is not None:
        ax.axvline(ref_S, color='k', ls='--', lw=1)
    if ref_B is not None:
        ax.axhline(ref_B, color='k', ls='--', lw=1)
    if ref_S is not None and ref_B is not None:
        ax.plot(ref_S, ref_B, 'kx', ms=8, mew=1.5, zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_{\rm ISCO}$  [g cm$^{-2}$]', fontsize=12)
    ax.set_ylabel(r'$B_{\rm ISCO}$  [G]', fontsize=12)
    ax.text(0.03, 0.97, f'N = {len(df_ok):,}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', fc='white', alpha=0.6))
    plt.tight_layout()
    return fig

In [ ]:
# una volta sola — arricchisce il dataframe
all_nt[1] = add_isco_values(all_nt[1])

# poi il plot è istantaneo, nessun ricalcolo
fig = scatter_bisco_sigma(all_nt[1], stage='pass_wkb', ref_B=1e4, ref_S=1e4)

#### radial profiles

In [ ]:
from AEI_plots.nt_radial_profiles_diagnostic import plot_nt_profiles

# uso base
fig, info = plot_nt_profiles(a=0.9, mdot=0.02, alpha=0.1)

# loop con salvataggio
for a in [0.0, 0.5, 0.9, -0.9]:
    fig, info = plot_nt_profiles(a=a, mdot=0.02, alpha=0.1,
                                 save=True, outdir='figures')
    plt.close(fig)   # libera memoria nel loop

#### spin

In [ ]:
from scipy.stats import binned_statistic_2d

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  HEATMAP  (a × param_x)  — risponde a "quali spin sono validi?"
# ═══════════════════════════════════════════════════════════════════════════════

def plot_spin_validity_heatmap(df, stage='pass_wkb',
                               param_x='mdot',
                               n_bins_a=40, n_bins_x=40,
                               figsize=(6, 4.5),
                               outdir='.', tag=''):
    """
    Heatmap 2D: asse x = param_x, asse y = spin a.
    Colore = frazione di valori del *terzo* parametro che passano `stage`.

    Esempio NT:   param_x='mdot'   → media su alpha
                  param_x='alpha'  → media su mdot
    Esempio Simple: param_x='B00'  → media su Sigma0
                    param_x='Sigma0' → media su B00

    Parameters
    ----------
    df      : DataFrame  già calcolato (all_nt[mm] o all_simple[cfg])
    stage   : str        colonna booleana ('pass_shear'/'pass_beta'/'pass_wkb')
    param_x : str        colonna da usare sull'asse x
    n_bins_a, n_bins_x : int  numero di bin
    figsize : tuple
    outdir  : str
    tag     : str        suffisso per il nome del file

    Returns
    -------
    fig : Figure
    stat : 2D array  (fraction map, shape n_bins_x × n_bins_a)
    """
    if stage not in df.columns:
        raise ValueError(f"Colonna '{stage}' non trovata nel DataFrame.")
    if param_x not in df.columns:
        raise ValueError(f"Colonna '{param_x}' non trovata nel DataFrame.")

    x_vals = df[param_x].values.astype(float)
    a_vals = df['a'].values.astype(float)
    f_vals = df[stage].values.astype(float)

    # scala logaritmica automatica se il range supera 2 decadi
    def _log(v):
        pos = v[v > 0]
        return len(pos) > 0 and np.log10(pos.max() / pos.min()) > 2

    log_x = _log(x_vals)
    xv = np.log10(np.where(x_vals > 0, x_vals, np.nan)) if log_x else x_vals

    # bin edges
    x_edges = np.linspace(np.nanmin(xv), np.nanmax(xv), n_bins_x + 1)
    a_edges = np.linspace(a_vals.min(), a_vals.max(), n_bins_a + 1)

    # binned mean of pass-flag
    valid = np.isfinite(xv) & np.isfinite(a_vals)
    stat, xe, ae, _ = binned_statistic_2d(
        xv[valid], a_vals[valid], f_vals[valid],
        statistic='mean',
        bins=[x_edges, a_edges],
    )

    # x-tick labels (convert back from log if needed)
    x_mids = 0.5 * (xe[:-1] + xe[1:])
    a_mids = 0.5 * (ae[:-1] + ae[1:])

    fig, ax = plt.subplots(figsize=figsize)

    im = ax.pcolormesh(xe, ae, stat.T,
                       vmin=0, vmax=1, cmap='RdYlGn',
                       shading='flat', rasterized=True)
    cb = plt.colorbar(im, ax=ax, pad=0.02)
    cb.set_label(f'Fraction passing  [{stage}]', fontsize=10)

    # ISCO boundary:  a = 0  dashed reference
    ax.axhline(0, color='white', ls=':', lw=1, alpha=0.6,
               label='a = 0  (Schwarzschild)')

    ax.set_xlabel(
        rf'$\log_{{10}}({param_x})$' if log_x else param_x,
        fontsize=11)
    ax.set_ylabel(r'Spin  $a$', fontsize=11)

    stage_label = {'pass_shear': 'shear',
                   'pass_beta':  '+β',
                   'pass_wkb':   '+WKB'}.get(stage, stage)
    ax.set_title(
        rf'Valid spin vs {param_x}  [{stage_label}]'
        '\n'
        r'colour = fraction of other parameters passing',
        fontsize=10
    )
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(False)

    plt.tight_layout()
    fname = f'{outdir}/spin_validity_{param_x}_{stage}{("_"+tag) if tag else ""}'
    for ext in ('pdf', 'png'):
        plt.savefig(fname + '.' + ext, bbox_inches='tight', dpi=150)
        print(f'  → {fname}.{ext}')
    plt.show()
    return fig, stat


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  VALID SPIN RANGE  —  per ogni cella (param1, param2)
# ═══════════════════════════════════════════════════════════════════════════════

def _valid_spin_stats(group, stage):
    """
    Per un gruppo di righe con la stessa (mdot, alpha) o (B00, Sigma0),
    restituisce statistiche sugli spin validi.
    """
    valid_a = group.loc[group[stage], 'a'].values
    if len(valid_a) == 0:
        return dict(a_med=np.nan, a_min=np.nan, a_max=np.nan,
                    a_width=np.nan, frac_pro=np.nan, frac_ret=np.nan,
                    n_valid=0)
    n = len(group)
    return dict(
        a_med   = float(np.median(valid_a)),
        a_min   = float(valid_a.min()),
        a_max   = float(valid_a.max()),
        a_width = float(valid_a.max() - valid_a.min()),
        frac_pro= float((valid_a > 0).sum() / n),   # over total spins
        frac_ret= float((valid_a < 0).sum() / n),
        n_valid = len(valid_a),
    )


def plot_valid_spin_range(df, stage='pass_wkb',
                          param1='mdot', param2='alpha',
                          n_bins=35,
                          figsize=(14, 4.2),
                          outdir='.', tag=''):
    """
    3 heatmap affiancate nel piano (param1, param2):

        Panel 1 — spin mediano delle soluzioni valide
                  Risponde a: "per questa (mdot,α), il AEI favorisce spin alto o basso?"

        Panel 2 — larghezza dell'intervallo valido  a_max − a_min
                  Risponde a: "il vincolo su spin è stretto o largo?"

        Panel 3 — frazione di spin prograde (a > 0) validi
                  Risponde a: "le soluzioni sono principalmente prograde?"

    Parameters
    ----------
    df      : DataFrame  già calcolato
    stage   : str        colonna booleana del filtro
    param1  : str        colonna asse x  (es. 'mdot' o 'B00')
    param2  : str        colonna asse y  (es. 'alpha' o 'Sigma0')
    n_bins  : int        bin per asse (stesso per x e y)
    figsize : tuple
    outdir  : str
    tag     : str        suffisso file

    Returns
    -------
    fig      : Figure
    df_stats : DataFrame  con una riga per cella (param1, param2)
    """
    for col in (stage, param1, param2, 'a'):
        if col not in df.columns:
            raise ValueError(f"Colonna '{col}' non trovata.")

    # ── aggrega per cella (param1, param2) ───────────────────────────────
    records = []
    for (p1, p2), grp in df.groupby([param1, param2]):
        s = _valid_spin_stats(grp, stage)
        records.append({param1: p1, param2: p2, **s})
    df_stats = pd.DataFrame(records)

    # ── scala log automatica ──────────────────────────────────────────────
    def _should_log(col):
        v = df_stats[col].values.astype(float)
        pos = v[v > 0]
        return len(pos) > 0 and np.log10(pos.max() / pos.min()) > 1.5

    log1 = _should_log(param1)
    log2 = _should_log(param2)

    p1v = np.log10(df_stats[param1].values.astype(float)) if log1 else df_stats[param1].values.astype(float)
    p2v = np.log10(df_stats[param2].values.astype(float)) if log2 else df_stats[param2].values.astype(float)

    e1 = np.linspace(np.nanmin(p1v), np.nanmax(p1v), n_bins + 1)
    e2 = np.linspace(np.nanmin(p2v), np.nanmax(p2v), n_bins + 1)

    def _bin2d(z_col, statistic='mean'):
        ok = np.isfinite(p1v) & np.isfinite(p2v) & np.isfinite(df_stats[z_col].values)
        stat, _, _, _ = binned_statistic_2d(
            p1v[ok], p2v[ok], df_stats[z_col].values[ok],
            statistic=statistic, bins=[e1, e2],
        )
        return stat   # shape (n_bins, n_bins), rows=param1, cols=param2

    # ── le tre mappe ─────────────────────────────────────────────────────
    panels = [
        ('a_med',    r'Median valid spin  $\langle a \rangle$',
         plt.cm.RdBu_r, Normalize(-1, 1)),
        ('a_width',  r'Valid spin width  $a_{\max} - a_{\min}$',
         plt.cm.viridis, Normalize(0, 2)),
        ('frac_pro', r'Fraction prograde  ($a > 0$) passing',
         plt.cm.RdYlGn, Normalize(0, 1)),
    ]

    xlabel = (rf'$\log_{{10}}(\mathrm{{{param1}}})$' if log1
              else param1.replace('_', r'\_'))
    ylabel = (rf'$\log_{{10}}(\mathrm{{{param2}}})$' if log2
              else param2.replace('_', r'\_'))

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    for ax, (col, title, cmap, norm) in zip(axes, panels):
        stat = _bin2d(col)
        im = ax.pcolormesh(e1, e2, stat.T,
                           cmap=cmap, norm=norm,
                           shading='flat', rasterized=True)
        cb = plt.colorbar(im, ax=ax, pad=0.02)

        # annotate cells with n_valid = 0 (grey)
        ax.set_facecolor('#1a1a2e')

        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=10)
        ax.grid(False)

        # contour at a_med = 0 (Schwarzschild boundary) for panel 1
        if col == 'a_med':
            try:
                ax.contour(0.5*(e1[:-1]+e1[1:]),
                           0.5*(e2[:-1]+e2[1:]),
                           stat.T,
                           levels=[0.0], colors='white',
                           linewidths=1.4, linestyles='--')
            except Exception:
                pass

    stage_label = {'pass_shear': 'shear only',
                   'pass_beta':  'shear + β',
                   'pass_wkb':   'all (+ WKB)'}.get(stage, stage)
    fig.suptitle(
        rf'Valid spin range in $({param1},\,{param2})$ space  —  {stage_label}',
        fontsize=11, y=1.02
    )
    plt.tight_layout()

    fname = f'{outdir}/spin_range_{param1}_{param2}_{stage}{("_"+tag) if tag else ""}'
    for ext in ('pdf', 'png'):
        plt.savefig(fname + '.' + ext, bbox_inches='tight', dpi=150)
        print(f'  → {fname}.{ext}')
    plt.show()
    return fig, df_stats


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  CONVENIENCE WRAPPER  —  pipeline completa per NT e Simple
# ═══════════════════════════════════════════════════════════════════════════════

def run_spin_analysis_nt(df_nt, stage='pass_wkb',
                         outdir='.', n_bins=35, tag='NT'):
    """
    Pipeline per il modello NT.

    Produce:
      • (a vs mdot) heatmap  — media su alpha
      • (a vs alpha) heatmap — media su mdot
      • valid spin range in (mdot, alpha) — 3 pannelli

    Parameters
    ----------
    df_nt  : DataFrame  — all_nt[mm]
    stage  : str
    outdir : str
    n_bins : int
    tag    : str
    """
    print(f"\n{'='*55}")
    print(f"  Spin analysis — NT model  |  stage = {stage}")
    print(f"{'='*55}")

    print("\n> (a vs mdot) heatmap...")
    plot_spin_validity_heatmap(df_nt, stage=stage, param_x='mdot',
                               n_bins_a=40, n_bins_x=40,
                               outdir=outdir, tag=tag)

    print("\n> (a vs alpha) heatmap...")
    plot_spin_validity_heatmap(df_nt, stage=stage, param_x='alpha',
                               n_bins_a=40, n_bins_x=40,
                               outdir=outdir, tag=tag)

    print("\n> Valid spin range in (mdot, alpha)...")
    fig, df_stats = plot_valid_spin_range(df_nt, stage=stage,
                                          param1='mdot', param2='alpha',
                                          n_bins=n_bins,
                                          outdir=outdir, tag=tag)
    return df_stats


def run_spin_analysis_simple(df_simple, stage='pass_wkb',
                             outdir='.', n_bins=35, tag='Simple'):
    """
    Pipeline per il modello Simple.

    Produce:
      • (a vs B00) heatmap   — media su Sigma0
      • (a vs Sigma0) heatmap — media su B00
      • valid spin range in (B00, Sigma0) — 3 pannelli
    """
    print(f"\n{'='*55}")
    print(f"  Spin analysis — Simple model  |  stage = {stage}")
    print(f"{'='*55}")

    print("\n> (a vs B00) heatmap...")
    plot_spin_validity_heatmap(df_simple, stage=stage, param_x='B00',
                               n_bins_a=40, n_bins_x=40,
                               outdir=outdir, tag=tag)

    print("\n> (a vs Sigma0) heatmap...")
    plot_spin_validity_heatmap(df_simple, stage=stage, param_x='Sigma0',
                               n_bins_a=40, n_bins_x=40,
                               outdir=outdir, tag=tag)

    print("\n> Valid spin range in (B00, Sigma0)...")
    fig, df_stats = plot_valid_spin_range(df_simple, stage=stage,
                                          param1='B00', param2='Sigma0',
                                          n_bins=n_bins,
                                          outdir=outdir, tag=tag)
    return df_stats

In [ ]:
df_stats_nt = run_spin_analysis_nt(all_nt[1], stage='pass_wkb',
                                    outdir='aei_2')

df_stats_s  = run_spin_analysis_simple(all_results[(1, 0.001)],
                                        stage='pass_wkb',
                                        outdir='aei_2')

#plot_spin_validity_heatmap(all_nt[1], stage='pass_wkb', param_x='mdot')
#plot_valid_spin_range(all_nt[1], stage='pass_wkb',
#                        param1='mdot', param2='alpha')